# 13 进程与线程

> 本笔记以 Jupyter Notebook 风格整理，系统梳理 Python 中的进程、线程、锁、队列、管道、进程池、线程池以及 GIL。

## 学习目标

- 理解并发、并行、同步、异步的区别
- 理解进程与线程的角色分工
- 掌握 `multiprocessing` 与 `threading` 的基本使用
- 掌握进程通信：`Queue`、`Pipe`
- 掌握进程池、线程池的常见 API
- 理解 GIL 的本质与多进程 / 多线程的选择策略

## 使用说明

1. 本笔记中的大多数线程示例可直接运行。
2. 部分多进程示例在 **Jupyter / Windows** 环境下可能需要保存为 `.py` 文件再运行，或者严格依赖 `if __name__ == '__main__':`。
3. 所有示例都尽量采用清晰、教学友好的写法，并增加了必要注释。

## 1. 一些核心概念

### 1.1 并发 vs 并行

**并发（Concurrency）**

- 在一段时间内，CPU 面对多个任务时，会把执行时间切成许多小片段，交替推进这些任务。
- 某一个瞬间，通常只有一个任务真正占用当前 CPU 核心。
- 给人的感觉像是“多个任务同时在做”，本质上是快速切换。

**并行（Parallelism）**

- 依赖多个 CPU 或多核心 CPU。
- 同一时刻，不同核心真的在执行不同任务。
- 这不是“像同时做”，而是“确实同时做”。

**一句话总结：**

- 并发：轮流推进多个任务
- 并行：同时执行多个任务

> 现代操作系统中，并发与并行通常会同时存在。

### 1.2 同步 vs 异步

**同步（sync）**

- 发起一个任务后，必须等待该任务完成，才能继续执行后续逻辑。
- 当前执行流会被阻塞。

**异步（async）**

- 发起一个任务后，不必等待它完成，就可以先去做别的事。
- 当前执行流不会因为这个任务而停住。
- 任务完成后，可以通过回调、事件、Future、消息等方式拿到结果。

**概念对比：**

- 并发 / 并行：描述任务“如何被执行”
- 同步 / 异步：描述任务“如何等待与组织”

**重要提醒：**

CPU 再快、核心再多，也不会改变你代码中的逻辑依赖关系。

例如：如果任务 1 → 任务 2 → 任务 3 被设计成严格同步关系，那么任务 1 没完成时，任务 2 就不能开始。

### 1.3 进程 vs 线程

**进程（Process）**

- 一个正在运行的程序，对应着一个或多个进程。
- 进程是操作系统进行**资源分配**的基本单位。
- 每个进程有自己独立的内存空间。

**线程（Thread）**

- 线程是进程内部的执行单元。
- 线程是操作系统进行 **CPU 调度** 的基本单位。
- 同一进程中的多个线程共享进程资源。

**记忆方式：**

- 进程更像“工厂”
- 线程更像“工厂里的工人”
- 工厂之间资源隔离，工厂内部工人共享资源

In [1]:
# 简单查看当前进程 ID 与父进程 ID
import os

print('当前进程 PID:', os.getpid())
print('父进程 PID:', os.getppid())

当前进程 PID: 31608
父进程 PID: 14888


## 2. 主进程与子进程

在操作系统中，每个进程都有唯一的 PID（进程编号）。

“主进程 / 子进程”是相对概念：

- A 创建 B，那么 B 是 A 的子进程
- B 再创建 C，那么 B 同时又是 C 的父进程

Python 中常用：

- `os.getpid()`：获取当前进程 ID
- `os.getppid()`：获取当前父进程 ID

Windows 下还可以用命令查看：

```bash
wmic process get Name,ParentProcessId,ProcessId
```

## 3. 使用 `Process` 创建进程

下面演示最基础的多进程写法。

**特别注意：**

- 在 Windows 中使用 `multiprocessing` 时，必须写 `if __name__ == '__main__':`
- 否则子进程启动时可能重复导入当前模块，造成递归创建进程的问题
- 在 Jupyter 中，这类代码有时不如在 `.py` 文件中运行稳定

In [2]:
import os
import time
from multiprocessing import Process

print('模块被加载:', __name__, 'PID =', os.getpid())

def speak():
    for index in range(5):
        print(f'我在说话 {index}, 进程 PID: {os.getpid()}, 父进程 PID: {os.getppid()}')
        time.sleep(1)

def study():
    for index in range(5):
        print(f'我在学习 {index}, 进程 PID: {os.getpid()}, 父进程 PID: {os.getppid()}')
        time.sleep(1)

if __name__ == '__main__':
    print('主进程开始执行')
    p1 = Process(target=speak)
    p2 = Process(target=study)

    p1.start()
    p2.start()

    print('主进程已提交两个子进程')

模块被加载: __main__ PID = 31608
主进程开始执行
主进程已提交两个子进程


## 4. `Process` 的常见参数

实例化 `Process` 时，常见参数如下：

- `group`：通常始终为 `None`
- `target`：子进程要执行的可调用对象
- `name`：进程名称
- `args`：位置参数元组
- `kwargs`：关键字参数字典
- `daemon`：是否设置为守护进程

当前进程名称可通过 `current_process().name` 查看。

In [3]:
import os
import time
from multiprocessing import Process, current_process

def speak(a, b, msg):
    for index in range(3):
        print(
            f'{msg} | a={a} | b={b} | 进程名={current_process().name} | '
            f'PID={os.getpid()} | PPID={os.getppid()} | 第{index}次发言'
        )
        time.sleep(1)

def study():
    for index in range(3):
        print(f'我在学习 {index}, PID={os.getpid()}, PPID={os.getppid()}')
        time.sleep(1)

if __name__ == '__main__':
    p1 = Process(
        target=speak,
        name='说话进程',
        args=(666, 888),
        kwargs={'msg': '示例消息'}
    )
    p2 = Process(target=study)

    p1.start()
    p2.start()

    p1.join()
    p2.join()

    print('主进程结束')

主进程结束


## 5. 进程控制

这一部分包括：

1. `Lock` / `RLock`
2. `join()`
3. `terminate()`
4. 守护进程 `daemon`

这些能力是编写稳定并发程序的基础。

In [4]:
# 5.1 Lock：避免多个进程同时操作同一资源导致混乱
import time
from multiprocessing import Process, Lock

def task_a(lock):
    for _ in range(3):
        with lock:
            print('任务A:', end=' ')
            print('完整', end=' ')
            print('输出', end=' ')
            print('一行')
        time.sleep(0.5)

def task_b(lock):
    for _ in range(3):
        with lock:
            print('任务B:', end=' ')
            print('也在', end=' ')
            print('完整', end=' ')
            print('输出')
        time.sleep(0.5)

if __name__ == '__main__':
    lock = Lock()
    p1 = Process(target=task_a, args=(lock,))
    p2 = Process(target=task_b, args=(lock,))

    p1.start()
    p2.start()
    p1.join()
    p2.join()

In [5]:
# 5.1 补充：RLock 允许同一个执行流重复上锁
import time
from multiprocessing import Process, RLock

def nested_lock_demo(lock):
    for _ in range(2):
        lock.acquire()
        lock.acquire()
        try:
            print('RLock 演示：同一进程内重复 acquire 不会立刻死锁')
        finally:
            lock.release()
            lock.release()
        time.sleep(0.5)

if __name__ == '__main__':
    lock = RLock()
    p = Process(target=nested_lock_demo, args=(lock,))
    p.start()
    p.join()

In [6]:
# 5.2 join：让当前进程等待目标进程结束
import os
import time
from multiprocessing import Process

def short_job():
    for i in range(3):
        print(f'short_job 第{i}次, PID={os.getpid()}')
        time.sleep(1)

def another_job():
    for i in range(3):
        print(f'another_job 第{i}次, PID={os.getpid()}')
        time.sleep(1)

if __name__ == '__main__':
    p1 = Process(target=short_job)
    p2 = Process(target=another_job)

    p1.start()
    p1.join(2)

    print('主进程等待 p1 2 秒后，继续启动 p2')
    p2.start()

    p1.join()
    p2.join()
    print('两个子进程都结束了')

主进程等待 p1 2 秒后，继续启动 p2
两个子进程都结束了


In [7]:
# 5.3 terminate：强制结束进程
import os
import time
from multiprocessing import Process

def long_job():
    try:
        for i in range(10):
            print(f'long_job 正在执行 {i}, PID={os.getpid()}')
            time.sleep(1)
    finally:
        print('注意：如果进程被 terminate，finally 不一定执行')

if __name__ == '__main__':
    p = Process(target=long_job)
    p.start()
    time.sleep(3)
    print('主进程准备终止 long_job')
    p.terminate()
    p.join()
    print('p.is_alive() ->', p.is_alive())

主进程准备终止 long_job
p.is_alive() -> False


In [8]:
# 5.4 守护进程 daemon
import os
import time
from multiprocessing import Process

def monitor():
    while True:
        try:
            with open('log_demo.txt', 'r', encoding='utf-8') as f:
                lines = sum(1 for _ in f)
        except FileNotFoundError:
            lines = 0
        print(f'守护进程 PID={os.getpid()}，当前日志行数={lines}')
        time.sleep(1)

def worker():
    for i in range(5):
        print(f'普通子进程工作中 {i}, PID={os.getpid()}')
        time.sleep(1)

if __name__ == '__main__':
    p1 = Process(target=monitor, daemon=True)
    p2 = Process(target=worker)

    p1.start()
    p2.start()

    with open('log_demo.txt', 'a', encoding='utf-8') as f:
        for i in range(5):
            f.write(f'日志记录 {i}\n')
            f.flush()
            time.sleep(1)

    p2.join()
    print('主进程结束后，守护进程也会结束')

主进程结束后，守护进程也会结束


## 6. 进程之间不共享变量

进程之间拥有各自独立的内存空间，因此普通变量不会天然共享。

这也是多进程安全的重要原因之一：

- 优点：天然隔离，不容易互相污染内存
- 缺点：数据传递没有线程那样直接，需要专门通信机制

In [9]:
# 验证 1：全局变量在不同进程中互不共享
from multiprocessing import Process

num = 100
names = []

def test1():
    global num, names
    num += 10
    names.append('张三')
    print('test1 ->', num, names)

def test2():
    global num, names
    num -= 10
    names.append('李四')
    print('test2 ->', num, names)

if __name__ == '__main__':
    p1 = Process(target=test1)
    p2 = Process(target=test2)
    p1.start()
    p2.start()
    p1.join()
    p2.join()
    print('主进程看到的最终值 ->', num, names)

主进程看到的最终值 -> 100 []


In [10]:
# 验证 2：即使通过参数传入，子进程对普通对象的修改也不会反映回主进程
from multiprocessing import Process

def test1(num, names):
    num += 10
    names.append('张三')
    print('test1 ->', num, names)

def test2(num, names):
    num -= 10
    names.append('李四')
    print('test2 ->', num, names)

if __name__ == '__main__':
    num = 100
    names = []

    p1 = Process(target=test1, args=(num, names))
    p2 = Process(target=test2, args=(num, names))

    p1.start()
    p2.start()
    p1.join()
    p2.join()

    print('主进程看到的最终值 ->', num, names)

主进程看到的最终值 -> 100 []


## 7. `Queue`（队列）

队列是一种 **先进先出（FIFO）** 的数据结构。

在多进程编程中，`multiprocessing.Queue` 是非常常见的通信手段。

常见方法：

- `put()`：入队
- `get()`：出队
- `empty()`：是否为空
- `full()`：是否已满
- `qsize()`：当前元素数量（某些平台可能不完全可靠）
- `put_nowait()` / `get_nowait()`：非阻塞模式

In [11]:
import queue
from multiprocessing import Queue

q1 = Queue()
q2 = Queue(3)

# put / get
q1.put(10)
q1.put(20)
q1.put(30)

print(q1.get())
print(q1.get())
print(q1.get())
print('q1 是否为空:', q1.empty())

# full / qsize
q2.put(100)
q2.put(200)
q2.put(300)
print('q2 是否已满:', q2.full())
print('q2 当前大小:', q2.qsize())

# 非阻塞读取示例
try:
    print('读取一个元素:', q2.get_nowait())
except queue.Empty:
    print('队列为空')

10
20
30
q1 是否为空: True
q2 是否已满: True
q2 当前大小: 3
读取一个元素: 100


In [ ]:
# 演示：队列满了以后，put 会等待；当有进程取走元素后，put 才继续
import time
from multiprocessing import Queue, Process

def consume_one(q):
    time.sleep(3)
    value = q.get()
    print('子进程取出了元素:', value)

if __name__ == '__main__':
    q = Queue(2)
    q.put('alpha')
    q.put('beta')
    print('队列是否已满:', q.full())

    p = Process(target=consume_one, args=(q,))
    p.start()

    print('准备向已满队列继续放入一个元素，此处会等待...')
    q.put('gamma')
    print('成功放入 gamma')

    print(q.get())
    print(q.get())
    p.join()

队列是否已满: True
准备向已满队列继续放入一个元素，此处会等待...


## 8. 使用 `Queue` 实现进程通信

核心思想：

- 一个进程负责生产数据
- 另一个进程负责消费数据
- 中间使用 `Queue` 传递信息

这是一种非常典型的生产者 / 消费者模式。

In [1]:
import time
from multiprocessing import Process, Queue

def producer(q):
    for index in range(5):
        print(f'生产者放入: {index}')
        q.put(index)
        time.sleep(0.5)

def consumer(q):
    for _ in range(5):
        data = q.get()
        print(f'消费者取出: {data}')
        time.sleep(1)

if __name__ == '__main__':
    q = Queue()
    p1 = Process(target=producer, args=(q,))
    p2 = Process(target=consumer, args=(q,))

    p1.start()
    p2.start()
    p1.join()
    p2.join()

    print('队列通信示例结束')

队列通信示例结束


## 9. 使用 `Pipe` 实现进程通信

`Pipe` 可以理解成一根“管道”。

- 一端发送：`send()`
- 一端接收：`recv()`

`Pipe(duplex=True)` 表示双向通信，`Pipe(duplex=False)` 表示单向通信。

In [2]:
import time
from multiprocessing import Process, Pipe

def sender(conn):
    time.sleep(2)
    conn.send({'value': 100, 'msg': '来自发送端的数据'})
    print('sender 已发送数据')

def receiver(conn):
    data = conn.recv()
    print('receiver 收到数据:', data)

if __name__ == '__main__':
    conn1, conn2 = Pipe(duplex=False)
    p1 = Process(target=sender, args=(conn1,))
    p2 = Process(target=receiver, args=(conn2,))

    p1.start()
    p2.start()
    p1.join()
    p2.join()

## 10. 继承 `Process` 类创建进程

当子进程逻辑比较复杂时，可以采用面向对象风格：

- 继承 `Process`
- 重写 `run()` 方法
- 依然使用 `start()` 启动，而不是手动调用 `run()`

In [3]:
import os
import time
from multiprocessing import Process

class SpeakProcess(Process):
    def __init__(self, a, b, **kwargs):
        super().__init__(**kwargs)
        self.a = a
        self.b = b

    def run(self):
        for index in range(3):
            print(f'SpeakProcess -> a={self.a}, b={self.b}, 第{index}次, PID={os.getpid()}')
            time.sleep(1)

class StudyProcess(Process):
    def run(self):
        for index in range(3):
            print(f'StudyProcess -> 第{index}次, PID={os.getpid()}')
            time.sleep(1)

if __name__ == '__main__':
    p1 = SpeakProcess(100, 200)
    p2 = StudyProcess()

    p1.start()
    p2.start()
    p1.join()
    p2.join()

    print('继承 Process 的示例结束')

继承 Process 的示例结束


## 11. 进程池 `ProcessPoolExecutor`

如果每来一个任务就创建一个新进程，会有较大开销。

进程池的优势：

- 提前创建固定数量的进程
- 重复利用这些进程执行任务
- 降低频繁创建 / 销毁进程的成本

In [4]:
# 11.1 submit + shutdown
import os
import time
from concurrent.futures import ProcessPoolExecutor

def work(n):
    print(f'任务 {n} 正在执行, PID={os.getpid()}')
    time.sleep(1)

if __name__ == '__main__':
    print('start')
    executor = ProcessPoolExecutor(3)
    for i in range(1, 8):
        executor.submit(work, i)
    executor.shutdown(wait=True)
    print('end')

start
end


In [3]:
from joblib import Parallel, delayed
import os, time

def work(n):
    print(f'任务 {n} 执行中, PID={os.getpid()}')
    time.sleep(1)
    return f'任务 {n} 的结果'

# 直接运行，不需要 if __name__ == '__main__'
results = Parallel(n_jobs=3)(delayed(work)(i) for i in range(1, 8))
for r in results:
    print(r)

任务 1 的结果
任务 2 的结果
任务 3 的结果
任务 4 的结果
任务 5 的结果
任务 6 的结果
任务 7 的结果


In [7]:
# 11.3 as_completed：按完成顺序取结果
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

def work(n):
    print(f'任务 {n} 执行中, PID={os.getpid()}')
    if n == 1:
        time.sleep(5)
    elif n == 2:
        time.sleep(3)
    else:
        time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    result_list = []
    with ProcessPoolExecutor(3) as executor:
        futures = [executor.submit(work, i) for i in range(1, 8)]
        for f in as_completed(futures):
            result_list.append(f.result())
    print(result_list)

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
# 11.4 add_done_callback
import os
import time
from concurrent.futures import ProcessPoolExecutor

def work(n):
    print(f'任务 {n} 执行中, PID={os.getpid()}')
    time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    result_list = []

    def done_func(future):
        result_list.append(future.result())

    with ProcessPoolExecutor(3) as executor:
        for i in range(1, 8):
            f = executor.submit(work, i)
            f.add_done_callback(done_func)

    print(result_list)

In [ ]:
# 11.5 map：批量提交任务，结果顺序与提交顺序一致
import os
import time
from concurrent.futures import ProcessPoolExecutor

def work(n):
    print(f'任务 {n} 执行中, PID={os.getpid()}')
    if n == 1:
        time.sleep(5)
    elif n == 2:
        time.sleep(3)
    else:
        time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    with ProcessPoolExecutor(3) as executor:
        results = executor.map(work, [1, 2, 3, 4, 5, 6, 7])
        print(list(results))

## 12. 使用 `Thread` 创建线程

任何一个正在运行的 Python 程序，至少有一个线程：**主线程**。

线程属于进程内部：

- 一个进程至少有一个线程
- 一个进程可以有多个线程
- 同一进程中的线程共享内存空间

线程在处理 I/O 密集型任务时通常很有价值。

In [ ]:
import os
import time
from threading import Thread, RLock, get_native_id

def speak(lock):
    for index in range(5):
        with lock:
            print(f'我在说话 {index}, 进程 PID={os.getpid()}, 线程 ID={get_native_id()}')
        time.sleep(1)

def study(lock):
    for index in range(5):
        with lock:
            print(f'我在学习 {index}, 进程 PID={os.getpid()}, 线程 ID={get_native_id()}')
        time.sleep(1)

if __name__ == '__main__':
    print(f'主线程启动, 进程 PID={os.getpid()}, 线程 ID={get_native_id()}')
    lock = RLock()
    t1 = Thread(target=speak, args=(lock,))
    t2 = Thread(target=study, args=(lock,))

    t1.start()
    t2.start()
    t1.join()
    t2.join()

    print('线程示例结束')

## 13. 继承 `Thread` 创建线程

和继承 `Process` 类似，线程也可以通过继承 `Thread` 的方式封装。

In [ ]:
import os
import time
from threading import Thread, RLock, get_native_id

class SpeakThread(Thread):
    def __init__(self, lock, **kwargs):
        super().__init__(**kwargs)
        self.lock = lock

    def run(self):
        for index in range(3):
            with self.lock:
                print(f'SpeakThread -> {index}, 进程 PID={os.getpid()}, 线程 ID={get_native_id()}')
            time.sleep(1)

class StudyThread(Thread):
    def __init__(self, lock, **kwargs):
        super().__init__(**kwargs)
        self.lock = lock

    def run(self):
        for index in range(3):
            with self.lock:
                print(f'StudyThread -> {index}, 进程 PID={os.getpid()}, 线程 ID={get_native_id()}')
            time.sleep(1)

if __name__ == '__main__':
    lock = RLock()
    t1 = SpeakThread(lock)
    t2 = StudyThread(lock)

    t1.start()
    t2.start()
    t1.join()
    t2.join()

    print('继承 Thread 的示例结束')

## 14. 线程池 `ThreadPoolExecutor`

线程池的用法与进程池非常相似。

常见能力：

- `submit()`
- `shutdown()`
- `result()`
- `as_completed()`
- `add_done_callback()`
- `map()`
- `with` 自动回收

In [ ]:
# 14.1 submit + shutdown
import time
from threading import RLock, get_native_id
from concurrent.futures import ThreadPoolExecutor

def work(n, lock):
    with lock:
        print(f'任务 {n} 在线程中执行, 线程 ID={get_native_id()}')
    time.sleep(1)

if __name__ == '__main__':
    executor = ThreadPoolExecutor(3)
    lock = RLock()
    for i in range(1, 8):
        executor.submit(work, i, lock)
    executor.shutdown(wait=True)
    print('线程池执行完毕')

In [ ]:
# 14.2 获取返回结果
import time
from threading import RLock, get_native_id
from concurrent.futures import ThreadPoolExecutor

def work(n, lock):
    with lock:
        print(f'任务 {n} 在线程中执行, 线程 ID={get_native_id()}')
    time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    lock = RLock()
    with ThreadPoolExecutor(3) as executor:
        futures = [executor.submit(work, i, lock) for i in range(1, 8)]
    for f in futures:
        print(f.result())

In [ ]:
# 14.3 as_completed
import time
from threading import RLock, get_native_id
from concurrent.futures import ThreadPoolExecutor, as_completed

def work(n, lock):
    with lock:
        print(f'任务 {n} 在线程中执行, 线程 ID={get_native_id()}')
    if n == 1:
        time.sleep(5)
    elif n == 2:
        time.sleep(3)
    else:
        time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    result_list = []
    lock = RLock()
    with ThreadPoolExecutor(3) as executor:
        futures = [executor.submit(work, i, lock) for i in range(1, 8)]
        for f in as_completed(futures):
            result_list.append(f.result())
    print(result_list)

In [ ]:
# 14.4 add_done_callback
import time
from threading import RLock, get_native_id
from concurrent.futures import ThreadPoolExecutor

def work(n, lock):
    with lock:
        print(f'任务 {n} 在线程中执行, 线程 ID={get_native_id()}')
    time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    result_list = []
    lock = RLock()

    def done_func(f):
        result_list.append(f.result())

    with ThreadPoolExecutor(3) as executor:
        for i in range(1, 8):
            future = executor.submit(work, i, lock)
            future.add_done_callback(done_func)

    print(result_list)

In [ ]:
# 14.5 map + with
import time
from threading import RLock, get_native_id
from concurrent.futures import ThreadPoolExecutor

def work(n, lock):
    with lock:
        print(f'任务 {n} 在线程中执行, 线程 ID={get_native_id()}')
    if n == 1:
        time.sleep(5)
    elif n == 2:
        time.sleep(3)
    else:
        time.sleep(1)
    return f'任务 {n} 的结果'

if __name__ == '__main__':
    with ThreadPoolExecutor(3) as executor:
        lock = RLock()
        result = executor.map(work, [1, 2, 3, 4, 5, 6, 7], [lock] * 7)
        print(list(result))

## 15. GIL（全局解释器锁）

### 15.1 什么是 GIL

GIL 是 CPython 解释器中的一把互斥锁。

它的核心作用是：

- 在同一进程内的多个线程中
- 某一时刻只允许一个线程执行 Python 字节码

因此，**CPython 中的多线程通常更偏向“并发”，而不是 CPU 意义上的真正并行”**。

### 15.2 为什么会有 GIL

主要是为了保证解释器级别的数据安全，尤其是对象引用计数等底层机制。

### 15.3 GIL 何时释放

- 主动释放：遇到 I/O 操作时
- 被迫释放：时间片切换时

### 15.4 GIL 与 `Lock` / `RLock` 的区别

- GIL：解释器层面的锁
- `Lock` / `RLock`：业务逻辑层面的锁

也就是说：

- GIL 不是为了让你的业务逻辑正确
- 业务中的共享数据、共享资源，仍然需要你自己加锁

In [ ]:
# GIL 不等于业务锁：下面用 RLock 保证打印完整性
import time
from threading import Thread, RLock, current_thread

def show_info1(lock):
    for _ in range(5):
        with lock:
            print('数', end='')
            print('据', end='')
            print('A')

def show_info2(lock):
    for _ in range(5):
        with lock:
            print('数', end='')
            print('据', end='')
            print('B')

if __name__ == '__main__':
    lock = RLock()
    t1 = Thread(target=show_info1, args=(lock,))
    t2 = Thread(target=show_info2, args=(lock,))
    t1.start()
    t2.start()
    t1.join()
    t2.join()

In [ ]:
# 使用 RLock 保护共享票号，避免多个线程卖出同一张票
import time
from threading import Thread, RLock, current_thread

current_ticket = 1

def sale(lock):
    global current_ticket
    while True:
        with lock:
            if current_ticket <= 20:
                print(f'{current_thread().name} 售出了第 {current_ticket} 张票')
                current_ticket += 1
            else:
                print('票已售空')
                break
        time.sleep(0.2)

if __name__ == '__main__':
    lock = RLock()
    t1 = Thread(target=sale, name='窗口1', args=(lock,))
    t2 = Thread(target=sale, name='窗口2', args=(lock,))
    t3 = Thread(target=sale, name='窗口3', args=(lock,))
    t1.start()
    t2.start()
    t3.start()
    t1.join()
    t2.join()
    t3.join()

## 16. 多进程 vs 多线程，该如何选择？

### 更适合多进程的场景

- CPU 密集型任务
- 例如：大规模数值计算、图像处理、视频编码、复杂模型推理前处理

原因：

- 多进程可以绕开单个进程中的 GIL 限制
- 能更充分利用多核 CPU

### 更适合多线程的场景

- I/O 密集型任务
- 例如：网络请求、文件读写、数据库访问、日志采集

原因：

- I/O 等待期间线程可切换
- 线程创建与切换开销通常比进程小

### 经验总结

- **算得多**：优先考虑多进程
- **等得多**：优先考虑多线程
- 如果任务模型更复杂，再结合异步、协程、消息队列等手段综合设计

In [ ]:
# CPU 密集型任务：更适合多进程
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def cpu_task(n):
    print(f'CPU 任务 {n} 开始')
    total = 0
    for i in range(3_000_000):
        total += i * i
    return total

if __name__ == '__main__':
    print('===== 多进程处理 CPU 密集型任务 =====')
    start = time.time()
    with ProcessPoolExecutor(4) as executor:
        list(executor.map(cpu_task, [1, 2, 3, 4]))
    end = time.time() - start
    print(f'多进程耗时: {end:.2f} 秒')

    # 若需要，也可对比多线程效果
    # print('===== 多线程处理 CPU 密集型任务 =====')
    # start = time.time()
    # with ThreadPoolExecutor(4) as executor:
    #     list(executor.map(cpu_task, [1, 2, 3, 4]))
    # end = time.time() - start
    # print(f'多线程耗时: {end:.2f} 秒')

In [ ]:
# I/O 密集型任务：更适合多线程
import os
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def copy_file(index):
    src_name = 'source_demo.bin'
    dst_name = f'source_demo_copy_{index}.bin'
    with open(src_name, 'rb') as src, open(dst_name, 'wb') as dst:
        while True:
            data = src.read(1024 * 1024)
            if not data:
                break
            dst.write(data)

if __name__ == '__main__':
    # 先创建一个测试文件
    if not os.path.exists('source_demo.bin'):
        with open('source_demo.bin', 'wb') as f:
            f.write(os.urandom(3 * 1024 * 1024))

    print('===== 多线程处理 I/O 密集型任务 =====')
    start = time.time()
    with ThreadPoolExecutor(4) as executor:
        for i in range(4):
            executor.submit(copy_file, i)
    end = time.time() - start
    print(f'多线程耗时: {end:.2f} 秒')

    # 如需对比多进程，可取消下方注释
    # print('===== 多进程处理 I/O 密集型任务 =====')
    # start = time.time()
    # with ProcessPoolExecutor(4) as executor:
    #     for i in range(4):
    #         executor.submit(copy_file, i)
    # end = time.time() - start
    # print(f'多进程耗时: {end:.2f} 秒')

# 总结

本笔记涵盖了 Python 并发编程中最核心的主题：

- 并发 / 并行
- 同步 / 异步
- 进程 / 线程
- 锁、队列、管道
- 进程池、线程池
- GIL
- 多进程与多线程的选择策略

如果把它们串起来理解，可以形成一条非常清晰的知识链：

1. 先理解概念差异
2. 再掌握创建并发单元的方法
3. 再掌握控制、通信、同步手段
4. 最后结合任务类型做技术选型

> 建议学习顺序：先线程，再进程；先基础 API，再线程池 / 进程池；最后再深入 GIL 与性能优化。